# 02 — Model Comparison
Compare ARIMA (local) vs XGBoost (global) forecasts side-by-side for any stored Wikipedia page.

**Prerequisites:** run the pipeline at least once so the XGBoost model is saved.

In [ ]:
import sys; sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from backend.data.data_loader import get_series
from backend.models import ARIMAForecaster, XGBoostGlobalForecaster
from backend.models.model_registry import load_model
from backend.services.forecast_service import generate_forecast
from backend.evaluation.metrics import compute_all, walk_forward_evaluate
from backend.utils.io_utils import init_db
from config import XGB_MODEL_PATH

init_db()

## Configuration

In [ ]:
PAGE    = 'Python'
HORIZON = 14        # days to forecast

## Load data & models

In [ ]:
df     = get_series(PAGE)
series = df['views']
print(f"Series length: {len(series)} days")

arima = ARIMAForecaster()
arima.fit(None)   # stateless — fits per call

xgb = load_model(XGB_MODEL_PATH)
print(f"XGBoost trained on {len(xgb.known_series)} series")

## Generate forecasts

In [ ]:
arima_fc = generate_forecast(arima, series, PAGE, HORIZON)
xgb_fc   = generate_forecast(xgb,   series, PAGE, HORIZON)

print('ARIMA forecast dates:', arima_fc.dates[:3], '...')
print('XGBoost forecast dates:', xgb_fc.dates[:3], '...')

## Forecast plot

In [ ]:
HIST_WINDOW = 90

def _rgba(hex_c, alpha):
    h = hex_c.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'rgba({r},{g},{b},{alpha})'

def _add_band(fig, fc, name, color):
    fig.add_trace(go.Scatter(x=fc.dates, y=fc.upper, mode='lines',
                             line=dict(width=0), showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=fc.dates, y=fc.lower, mode='lines',
                             line=dict(width=0), fill='tonexty',
                             fillcolor=_rgba(color, 0.15),
                             showlegend=False, hoverinfo='skip'))
    fig.add_trace(go.Scatter(x=fc.dates, y=fc.mean, name=name,
                             mode='lines+markers',
                             line=dict(color=color, width=2, dash='dash'),
                             marker=dict(size=4)))

fig = go.Figure()
hist = series.tail(HIST_WINDOW)
fig.add_trace(go.Scatter(x=hist.index.astype(str).tolist(), y=hist.values,
                         name=f'History (last {HIST_WINDOW}d)',
                         line=dict(color='#94a3b8', width=1.5)))

_add_band(fig, arima_fc, 'ARIMA',   '#60a5fa')
_add_band(fig, xgb_fc,   'XGBoost', '#f472b6')

fig.update_layout(
    title=f'{PAGE} — {HORIZON}-day Forecast: ARIMA vs XGBoost',
    template='plotly_dark', hovermode='x unified',
    xaxis_title='Date', yaxis_title='Daily views',
)
fig.show()

## Walk-forward evaluation

In [ ]:
def _arima_fn(train, h):
    return np.array(generate_forecast(arima, train, PAGE, h).mean)

def _xgb_fn(train, h):
    return np.array(generate_forecast(xgb, train, PAGE, h).mean)

arima_metrics = walk_forward_evaluate(series, _arima_fn, horizon=HORIZON, n_splits=3)
xgb_metrics   = walk_forward_evaluate(series, _xgb_fn,   horizon=HORIZON, n_splits=3)

results = pd.DataFrame({
    'ARIMA':   arima_metrics,
    'XGBoost': xgb_metrics,
}).T
results

In [ ]:
import plotly.express as px

fig = px.bar(
    results.reset_index().melt(id_vars='index', var_name='Metric', value_name='Value'),
    x='Metric', y='Value', color='index', barmode='group',
    title=f'Walk-forward Metrics — {PAGE}',
    template='plotly_dark',
    color_discrete_map={'ARIMA': '#60a5fa', 'XGBoost': '#f472b6'},
)
fig.show()